In [168]:
import pandas as pd
import great_tables as gt

df = pd.read_csv('/Users/natemekonen/Desktop/Data_Projects/nba_data_project/data/model_output/FINAL_SOPH_TABLE.csv', index_col=0)
df = df[['Images', 'Team_Logo', 'Player', 'Team', 'Position', 'Age', 'Allstar_Prob', 'Similar_Players']]
df = df.iloc[0:10, :]

df['Player_Team_Image'] = df.apply(
    lambda row: f"""
    <div style="position: relative; width: 90px; height: 80px;">

        <!-- team logo behind -->
        <img src="{row['Team_Logo']}"
             style="
                 position: absolute;
                 left: 0px;
                 top: 10px;
                 width: 50px;
                 opacity: 1.0;
                 z-index: 1;
             ">

        <!-- player image front -->
        <img src="{row['Images']}"
                style="
                 position: absolute;
                 left: 18px;
                 top: 0px;
                 width: 100px;
                 height: 100px;
                 object-fit: contain;
                 z-index: 2;
                 display: block;
             ">
    </div>
    """,
    axis=1
)

df = df[['Player_Team_Image', 'Player', 'Team', 'Position', 'Age', 'Allstar_Prob', 'Similar_Players']]

df['Similar_Players'] = df['Similar_Players'].str.replace(', ', '<br>').str.replace(',', '<br>')

soph_table = (
gt.GT(df)
    .opt_table_font(
        font=gt.google_font(name='Fira Sans'),
        weight='lighter'
    )

    .tab_header(
        title='Sophomore Projections'
    )

    .cols_label(
        Player_Team_Image = '',
        Allstar_Prob = 'All-Star %',
        Similar_Players = 'Similar Player Profiles'
    )

    .tab_style(
        style=gt.style.css("""
            text-align: center !important;
        """),
        locations=gt.loc.column_labels(
            columns=['Player_Team_Image', 'Team', 'Position', 'Age', 'Allstar_Prob', 'Similar_Players']
        )
    )
    .tab_style(
        style=gt.style.css("""
            font-weight: bold !important;
        """),
        locations=gt.loc.column_labels(columns=['Allstar_Prob'])
    )
    .tab_style(
        style=gt.style.text(weight='bold'),
        locations=gt.loc.body(columns='Allstar_Prob')
    )

    .tab_options(
        heading_title_font_size='28px',
        table_background_color='#F5EFEB' 
    )

    .cols_align(columns='Team', align='center')
    .cols_align(columns='Position', align='center')
    .cols_align(columns='Age', align='center')
    .cols_align(columns='Allstar_Prob', align='center')
    .cols_align(columns='Similar_Players',align='left')

    .cols_width(
    Team='60px',
    Position='60px',
    Age='60px'
    )

    .fmt_number(columns='Age',decimals=0)
    .fmt_percent(columns='Allstar_Prob',decimals=1)

    .fmt_markdown(columns='Similar_Players') 
    
    .data_color(
        columns='Allstar_Prob',
        palette=['#ff4d4d', '#ffffff', '#00b050'],
        domain=[-.75, 1]
    )

    .tab_source_note(
        source_note=gt.md('*Source: Basketball Reference + Custom Random Forest & KNN Models*')
    )
)

html = soph_table.as_raw_html()

with open("sophomore_projections.html", "w", encoding="utf-8") as f:
    f.write(html)